<center><font size="+4">Programming & Data Analytics & AI 1 2025/2026</font></center>
<center><font size="+2">Sant'Anna School of Advanced Studies, Pisa, Italy</font></center>
<center><img src="https://github.com/EMbeDS-education/ComputingDataAnalysisModeling20242025/raw/main/PDAI/jupyter/jupyterNotebooks/images/sssaLEMBEDSdtu.png" width="900" alt="L'EMbeDS"></center>

<center><font size="+2">Course responsible</font></center>
<center><font size="+2">Andrea Vandin a.vandin@santannapisa.it</font></center>
<center><font size="+2">Lecturer</font></center>
<center><font size="+2">Lorenzo Emer lorenzo.emer@santannapisa.it</font></center>

---

<center><font size="+4">Module 2, Part 3, Research-driven topics</font></center>
<center><font size="+4">Lecture 7: Generative AI and LLM</font></center>

---

This notebook demonstrates:
- System prompt + temperature
- Structured prompting (JSON)
- Zero-shot vs few-shot
- Chain-of-thought style prompting
- Retrieval-Augmented Generation (RAG)

In [1]:
# Cell 1 — Install dependencies
# We install Hugging Face Transformers + Accelerate for loading the model locally.
%pip install transformers accelerate torch sentencepiece

  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 4.5 MB/s  0:00:02.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 6.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 3.3 MB/s  0:00:01m 3.2 MB/s eta 0:00:01
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 2.7 MB/s  0:00:01m 2.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 2.9 MB/s  0:00:27 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 1.8 MB/s  0:00:001.8 MB/s eta 0:00:01:01
Using cached h

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert not torch.cuda.is_available(), "This notebook is configured for CPU; CUDA appears available."

/Users/leonardopratelli/Desktop/Programming-Data-Analytics-AI-II-personal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.10.0
CUDA available: False


# Load a small local instruct model (CPU)
We load an instruct-tuned model that can follow prompts reasonably well even on CPU.

In [3]:
# Cell 3 — Load model + tokenizer (CPU)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": "cpu"},   # force CPU
    torch_dtype=torch.float32 # CPU-friendly dtype
)

print("Loaded:", MODEL_ID)

`torch_dtype` is deprecated! Use `dtype` instead!
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

# Define a single `generate()` function
This helper:
- supports **system prompt** (high-level behavior)
- supports **temperature** (randomness)
- uses the model’s **chat template** when available (important for instruct models)

In [4]:
def generate(user_text, system=None, temperature=0.2, max_new_tokens=160):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_text})

    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = (system + "\n" if system else "") + user_text

    inputs = tokenizer(prompt, return_tensors="pt")

    input_length = inputs["input_ids"].shape[1]

    do_sample = temperature > 0
    out_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated_ids = out_ids[0][input_length:]
    output = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return output.strip()

# Sanity check
We run a simple prompt to confirm the model works.

In [7]:
# Cell 5 — Sanity check
print(generate("Explain inflation in 2 sentences.", temperature=0.2))

NameError: name 'model' is not defined

# Next-token prediction intuition
LLMs generate text by repeatedly predicting the **next token**.  
We demonstrate this with a short “continuation” prompt and **temperature=0** to reduce randomness.

In [8]:
# Cell 6 — Next-token prediction intuition
prompt = "The capital of France is"
print(generate(prompt, temperature=0.0, max_new_tokens=40))

NameError: name 'model' is not defined

# System prompt: same model, different behavior
A **system prompt** changes the model’s role/tone/constraints at inference time.  
We ask the **same question** with two different system prompts.

In [9]:
# Cell 7 — System prompt demo
question = "Explain inflation in 3–4 sentences."

out_lecturer = generate(
    question,
    system="You are a university lecturer. Be precise. Use one simple example.",
    temperature=0.2,
)

out_simple = generate(
    question,
    system="Explain like I'm 12 years old. Use very simple words and one example.",
    temperature=0.2,
)

print("=== Lecturer ===\n", out_lecturer, "\n")
print("=== Simple ===\n", out_simple)

NameError: name 'model' is not defined

# 6. Temperature: creativity vs stability
**Temperature** controls randomness in token selection:
- low temperature → more stable, conservative outputs
- higher temperature → more diverse, creative outputs (but also riskier)

In [10]:
# Cell 8 — Temperature sweep
prompt = "Give me 6 catchy names for a coffee shop."

for t in [0.0, 0.8, 1.1]:
    print(f"\n=== Temperature = {t} ===")
    print(generate(prompt, temperature=t, max_new_tokens=120))


=== Temperature = 0.0 ===


NameError: name 'model' is not defined

# 7. Structured prompting (output control)
Prompt engineering works best when we specify:
- role
- task
- constraints
- **output format**

Here we ask for a JSON-like structured answer.

In [ ]:
# Cell 9 — Structured prompt (JSON request)
text = "We launched a new app feature, but users complain it is slow and confusing."

structured_prompt = f"""
You are a product analyst.

Task:
Extract a structured analysis of the text.

Return valid JSON with EXACT keys:
- problem
- evidence
- hypothesis
- next_action

Text:
{text}
""".strip()

print(generate(structured_prompt, temperature=0.2, max_new_tokens=220))

```json
{
  "problem": "Users complain that the new app feature is slow and confusing.",
  "evidence": {
    "users_complain_slow_and_confusing": [
      "We launched a new app feature",
      "users complain it is slow and confusing"
    ]
  },
  "hypothesis": "The new app feature has performance issues and is difficult to understand for users.",
  "next_action": "Investigate the root cause of the slowdown and confusion, possibly through user testing or feedback collection."
}
```


# Zero-shot vs Few-shot
- **Zero-shot**: no examples, just instructions  
- **Few-shot**: we include example(s) showing the desired mapping

We compare both on the same sentiment task.

In [ ]:
# Cell 10 — Zero-shot vs Few-shot
zero_shot = """
Classify the sentiment as Positive, Neutral, or Negative.
Return ONLY one word.

Review: "The service was slow and the food was cold."
""".strip()

few_shot = """
Classify sentiment as Positive, Neutral, or Negative.
Return ONLY one word.

Review: "Amazing food and friendly staff."
Sentiment: Positive

Review: "It was okay, nothing special."
Sentiment: Neutral

Review: "The service was slow and the food was cold."
Sentiment:
""".strip()

print("Zero-shot →", generate(zero_shot, temperature=0.0, max_new_tokens=10))
print("Few-shot  →", generate(few_shot, temperature=0.0, max_new_tokens=10))

Zero-shot → Negative
Few-shot  → Negative


# Chain-of-thought style prompting (reasoning transparency)
For multi-step reasoning, we can ask the model to “think step-by-step”.  
This often improves performance and makes intermediate logic visible—  
but it does **not** guarantee correctness.

In [ ]:
# Cell 11 — Chain-of-thought style demo
cot_prompt = """
A train travels 120 km in 2 hours.
Explain your reasoning step by step, then provide the final answer in one line starting with 'Final:'.
""".strip()

print(generate(cot_prompt, temperature=0.0, max_new_tokens=200))

To determine how far the train travels per hour, we start by dividing the total distance traveled by the time taken.

Step 1: Total distance = 120 km
Step 2: Time taken = 2 hours

Now, calculate the speed:
\[ \text{Speed} = \frac{\text{Distance}}{\text{Time}} = \frac{120 \text{ km}}{2 \text{ hours}} = 60 \text{ km/hour} \]

Final: The train travels at a speed of 60 kilometers per hour.


# RAG over "Statuto Scuola Superiore Sant’Anna.pdf"
Pipeline:
1) Extract text from PDF
2) Chunk text
3) Embed chunks
4) Retrieve top-k chunks for a query
5) Inject retrieved chunks into local LLM prompt (grounded answer)

In [ ]:
%pip install pymupdf sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
PDF_PATH = "Statuto Scuola Superiore Sant’Anna.pdf"

import fitz  # pymupdf

def extract_pdf_text(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages = []
    for i in range(len(doc)):
        text = doc[i].get_text("text")
        pages.append({"page": i+1, "text": text})
    return pages

pages = extract_pdf_text(PDF_PATH)
print("Pages extracted:", len(pages))
print("Sample (page 1, first 500 chars):")
print(pages[0]["text"][:500])



Pages extracted: 28
Sample (page 1, first 500 chars):
STATUTO DELLA SCUOLA SUPERIORE DI STUDI UNIVERSITARI E DI PERFEZIONAMENTO SANT’ANNA 
 
 
 
STATUTO DELLA SCUOLA SUPERIORE DI STUDI UNIVERSITARI E 
DI PERFEZIONAMENTO SANT’ANNA 
 
Emanato con D.D. n.770 del 09.12.2011; 
integrato e modificato con D.R. n. 94 del 09.03.2015; 
integrato e modificato con D.R. n. 48 del 25.01.2018; 
integrato e modificato con D.R. n. 146 del 07.03.2022; 
integrato e modificato con D.R. n. 883 del 15.12.2023,   
                   pubblicato nella G.U. n. 301 del 28.12


In [ ]:
import re

def chunk_text(pages, chunk_chars=1200, overlap=200):
    chunks = []
    for p in pages:
        t = re.sub(r"\s+", " ", p["text"]).strip()
        if not t:
            continue
        start = 0
        while start < len(t):
            end = min(len(t), start + chunk_chars)
            chunk = t[start:end]
            chunks.append({
                "page": p["page"],
                "chunk": chunk
            })
            start = max(end - overlap, end)
    return chunks

chunks = chunk_text(pages)
print("Chunks:", len(chunks))
print("Example chunk:", chunks[0]["chunk"][:300], "...")

Chunks: 85
Example chunk: STATUTO DELLA SCUOLA SUPERIORE DI STUDI UNIVERSITARI E DI PERFEZIONAMENTO SANT’ANNA STATUTO DELLA SCUOLA SUPERIORE DI STUDI UNIVERSITARI E DI PERFEZIONAMENTO SANT’ANNA Emanato con D.D. n.770 del 09.12.2011; integrato e modificato con D.R. n. 94 del 09.03.2015; integrato e modificato con D.R. n. 48 d ...


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # fast CPU embedding model
chunk_texts = [c["chunk"] for c in chunks]

chunk_emb = embedder.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)
chunk_emb = np.array(chunk_emb, dtype=np.float32)

print("Embeddings:", chunk_emb.shape)

/Users/lorenzoemer/.venv_spacy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 103/103 [00:00<00:00, 1941.67it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|████████████████████████████████████| 3/3 [00:01<00:00,  2.99it/s]

Embeddings: (85, 384)


In [ ]:
def retrieve_top_k(query: str, k: int = 4):
    q = embedder.encode([query], normalize_embeddings=True)[0].astype(np.float32)
    sims = chunk_emb @ q
    idx = np.argsort(-sims)[:k]
    results = []
    for i in idx:
        results.append({
            "score": float(sims[i]),
            "page": chunks[i]["page"],
            "chunk": chunks[i]["chunk"]
        })
    return results

query = "What are the main governance bodies of the Scuola and what do they do?"
results = retrieve_top_k(query, k=4)

for r in results:
    print(f"score={r['score']:.3f} page={r['page']}  text={r['chunk'][:120]}...")

score=0.478 page=5  text=eccellenza e con la Scuola Normale Superiore 1. La Scuola promuove la collaborazione istituzionale con le altre Scuole a...
score=0.386 page=3  text=STATUTO DELLA SCUOLA SUPERIORE DI STUDI UNIVERSITARI E DI PERFEZIONAMENTO SANT’ANNA Art. 5 Sede e nome della Scuola 1. L...
score=0.381 page=7  text= nell’Albo della Scuola, salvo che essi non dispongano diversamente. Art. 19 Strumenti di programmazione 1. La Scuola ad...
score=0.358 page=18  text=i, in aree ritenute strategiche per le linee di ricerca della Scuola. 5. Gli Istituti e i Centri di ricerca interdiscipl...


In [ ]:
def rag_answer(question: str, retrieved, temperature=0.0):
    context_blocks = []
    for r in retrieved:
        context_blocks.append(f"[Page {r['page']}] {r['chunk']}")
    context = "\n\n".join(context_blocks)

    prompt = f"""
You are answering questions about the "Statuto Scuola Superiore Sant’Anna".
Use ONLY the provided excerpts. If the answer is not contained, say: "I don't know based on the provided excerpts."

Excerpts:
{context}

Question: {question}

Return:
1) Answer (short)
2) Evidence: list the page numbers you used
""".strip()

    return generate(prompt, temperature=temperature, max_new_tokens=220)

question = "Which bodies govern the Scuola and what are their roles?"
retrieved = retrieve_top_k(question, k=4)
print(rag_answer(question, retrieved, temperature=0.0))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1) The bodies that govern the Scuola include:

   - The Assemblea delle allieve e degli allievi (Assembly of Students and Graduates)
   - The Comitato Unico di Garanzia (Committee for Guarantee)

2) Evidence: Page 3, Pages 7-8


In [ ]:
def show_retrieved(retrieved):
    for r in retrieved:
        print("="*80)
        print(f"Score: {r['score']:.3f} | Page: {r['page']}")
        print(r["chunk"][:800])

retrieved = retrieve_top_k("What are the governance bodies of the Scuola?", k=4)
show_retrieved(retrieved)

Score: 0.483 | Page: 5
eccellenza e con la Scuola Normale Superiore 1. La Scuola promuove la collaborazione istituzionale con le altre Scuole a ordinamento speciale, predisponendo gli opportuni meccanismi di raccordo e di coordinamento. 2. La Scuola considera la Scuola Normale Superiore interlocutore privilegiato per le attività di ricerca, di formazione e di terza missione, valorizzando la collaborazione consolidata nella propria esperienza storica. Si coordina con essa per la definizione delle modalità di conferimento dei titoli di studio aventi valore legale.
Score: 0.394 | Page: 3
STATUTO DELLA SCUOLA SUPERIORE DI STUDI UNIVERSITARI E DI PERFEZIONAMENTO SANT’ANNA Art. 5 Sede e nome della Scuola 1. La Scuola ha la sede legale in Pisa e può utilizzare nei rapporti esterni ed interni la denominazione abbreviata “Scuola Superiore Sant’Anna o “Sant’Anna School of Advanced Studies”. 2. La Scuola ha sede centrale nell’edificio storico già sede del Conservatorio Sant’Anna. Può istituire o 